<div class="alert alert-block alert-danger">
Before you start running this notebook, create a Hail-enabled Dataproc cluster with a <b>high compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab Spark cluster</b> and, under <b>Software to install</b>, choose <b>Hail</b>.</li>
    <li>Configure a high-compute cluster (multiple workers). Cluster creation can take about 10 minutes.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a> and <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/cloud_app_types/dataproc/">Use Dataproc (Spark) and Hail</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook produces the ancestry PCA panel of Figure 1 in the AoU-LR manuscript. It stages precomputed ancestry predictions and loadings, includes PacBio, PacBio+ONT, and HPRC samples, and draws a scree plot plus 2D and 3D PC projections colored by genetic ancestry.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

## Set up environment

In [ ]:
import hail as hl
from hail.plot import show

In [ ]:
import os
import numpy as np
from collections import Counter

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits import mplot3d

In [ ]:
# Workspace details Terra sets as environment variables (empty strings when run elsewhere).
bucket = os.environ['WORKSPACE_BUCKET'] if 'WORKSPACE_BUCKET' in os.environ else ''
workspace = os.environ['WORKSPACE_NAME'] if 'WORKSPACE_NAME' in os.environ else ''
namespace = os.environ['WORKSPACE_NAMESPACE'] if 'WORKSPACE_NAMESPACE' in os.environ else ''

## List samples

### List PacBio samples with additional ONT data

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# Oxford Nanopore Phase 1 research IDs.
!gsutil cp {AOU_LR_DATA}/ont_ids.txt ont_ids.txt
ont_ids = [line.strip() for line in open("ont_ids.txt") if line.strip()]
len(ont_ids)

### List AoU+HPRC long-read samples

In [ ]:
# 1074_bams.csv lists every long-read BAM in the workspace (AoU PacBio plus HPRC); the first column is the sample ID.
all_ids = !gsutil cat {bucket}/bam_csv/1074_bams.csv | cut -d',' -f1
len(all_ids)

### List AoU long-read samples

In [ ]:
# AoU samples are the rows without 'HPRC' in them.
pb_ids = !gsutil cat {bucket}/bam_csv/1074_bams.csv | grep -v HPRC | cut -d',' -f1
len(pb_ids)

### List HPRC long-read samples

In [ ]:
hprc_ids = !gsutil cat {bucket}/bam_csv/1074_bams.csv | grep HPRC | cut -d',' -f1
len(hprc_ids)

## Stage data

### Expand AoU ancestry predictions tarball and stage

In [ ]:
# The AoU short-read ancestry pipeline's outputs (ancestry predictions and PCA loadings) come as Hail
# tables in tarballs. Unpack each one and copy it to the workspace bucket, where Hail can read it.
!gsutil -u $GOOGLE_PROJECT -m cp -r {AOU_LR_DATA}/delta_v1_gt_no_ext_aian_gq0_prod.ancestry_preds.ht.tar.gz .
!gunzip -c delta_v1_gt_no_ext_aian_gq0_prod.ancestry_preds.ht.tar.gz | tar xvf -
!gsutil -m cp -r delta_v1_gt_no_ext_aian_gq0_prod.ancestry_preds.ht {bucket}/scratch/

### Expand AoU PCA loadings tarball and stage

In [ ]:
!gsutil -u $GOOGLE_PROJECT -m cp -r {AOU_LR_DATA}/delta_v1_gt_no_ext_aian_gq0_prod_loadings.ht.tar.gz .
!gunzip -c delta_v1_gt_no_ext_aian_gq0_prod_loadings.ht.tar.gz | tar xvf -
!gsutil -m cp -r delta_v1_gt_no_ext_aian_gq0_prod_loadings.ht {bucket}/scratch/

### Download ancestry predictions

In [ ]:
# The same ancestry predictions as a TSV (per-participant predicted ancestry and PC scores).
!gsutil -u $GOOGLE_PROJECT -m cp -r {AOU_LR_DATA}/delta_v1_gt_no_ext_aian_gq0_prod.ancestry_preds.tsv .

### Download eigenvalues

In [ ]:
!gsutil cp {AOU_LR_DATA}/delta_PCA/delta_PCA.eigenvalues.ht delta_PCA.eigenvalues.txt

In [ ]:
# Proportion of variance explained by each PC of the reference PCA (the first eigenvalue line is skipped).
eigenvalues_txt = f'delta_PCA.eigenvalues.txt'

with open(eigenvalues_txt, 'r') as file:
    eigenvalues = [float(line.strip()) for line in file]
    
eigenvalues = eigenvalues[1:]
total_variance = sum(eigenvalues)
variance_explained = [eig / total_variance for eig in eigenvalues]

variance_explained

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(eigenvalues)+1), variance_explained, marker='o')
plt.title('Scree Plot')
plt.xlabel('Principal Component')
plt.ylabel('Proportion of Variance Explained')
plt.grid(True)
plt.show()

## Project HPRC samples into AoU's PCA space

In [ ]:
# Start Hail, using GRCh38 coordinates.
hl.init()

In [ ]:
hl.default_reference('GRCh38')

In [ ]:
# PCA loadings and allele frequencies of the reference panel, used to project new samples.
loadings = hl.read_table(f'{bucket}/scratch/delta_v1_gt_no_ext_aian_gq0_prod_loadings.ht')

In [ ]:
loadings.show(10)

In [ ]:
# Ancestry predictions for all short-read participants; pca_features holds each participant's PC scores.
ancestry_preds_tbl = hl.import_table(
    f'{AOU_LR_DATA}/delta_v1_gt_no_ext_aian_gq0_prod.ancestry_preds.tsv',
    types={ 'pca_features': hl.tarray(hl.tfloat64) }
)

ancestry_preds_tbl.show(10)

In [ ]:
# Predicted ancestry labels: all participants (l1), PacBio participants (l2), ONT participants (o2).
l1 = ancestry_preds_tbl.ancestry_pred_other.collect()

In [ ]:
ancestry_preds_pb_tbl = ancestry_preds_tbl.filter(hl.literal(pb_ids).contains(ancestry_preds_tbl.research_id))
l2 = ancestry_preds_pb_tbl.ancestry_pred_other.collect()

In [ ]:
ancestry_preds_ont_tbl = ancestry_preds_tbl.filter(hl.literal(ont_ids).contains(ancestry_preds_tbl.research_id))
o2 = ancestry_preds_ont_tbl.ancestry_pred_other.collect()

In [ ]:
# Project the 47 HPRC samples onto the AoU PCA. The import and projection (commented out) were run once;
# later runs read the saved projection.
hprc_gvcf = f'{AOU_LR_DATA}/HPRC_samples.g.vcf.bgz'

In [ ]:
# hprc_mt = hl.import_vcf(
#     hprc_gvcf,
#     array_elements_required=False,
#     force_bgz=True
# )

# hprc_mt.write(f'{bucket}/scratch/hprc-47.mt')

In [ ]:
#grch38_mt = hl.read_matrix_table(f'{bucket}/scratch/hprc-47.mt')
#testing_ht = hl.experimental.pc_project(grch38_mt.GT, loadings.loadings, loadings.af)
#testing_ht.write(f'{bucket}/scratch/HPRC.projected.ht', overwrite=True)

In [ ]:
testing_ht = hl.read_table(f'{bucket}/scratch/HPRC.projected.ht')

In [ ]:
testing_ht.show(5)

In [ ]:
# First three PC scores for each group: all participants, PacBio, ONT, and projected HPRC samples.
pc1 = ancestry_preds_tbl.pca_features[0].collect()
pc2 = ancestry_preds_tbl.pca_features[1].collect()
pc3 = ancestry_preds_tbl.pca_features[2].collect()

pb_pc1 = ancestry_preds_pb_tbl.pca_features[0].collect()
pb_pc2 = ancestry_preds_pb_tbl.pca_features[1].collect()
pb_pc3 = ancestry_preds_pb_tbl.pca_features[2].collect()

ont_pc1 = ancestry_preds_ont_tbl.pca_features[0].collect()
ont_pc2 = ancestry_preds_ont_tbl.pca_features[1].collect()
ont_pc3 = ancestry_preds_ont_tbl.pca_features[2].collect()

hprc_pc1 = testing_ht.scores[0].collect()
hprc_pc2 = testing_ht.scores[1].collect()
hprc_pc3 = testing_ht.scores[2].collect()

In [ ]:
len(pc1)

## Plot PCA

In [ ]:
# Colours for each continental ancestry group, and for the long-read sample markers.
color_map = {
    'afr': '#D573BE',
    'amr': '#519D3E',
    'eas': '#C53A32',
    'eur': '#3976AF',
    'mid': '#EF8536',
    'sas': '#866BE8',
    'oth': '#84584E'
}

grey_map = {
    'afr': '#CCCCCC',
    'amr': '#CCCCCC',
    'eas': '#CCCCCC',
    'eur': '#CCCCCC',
    'mid': '#CCCCCC',
    'sas': '#CCCCCC',
    'oth': '#CCCCCC'
}

outer_color = "#333333"

pb_color = "#FFFFFF"
pb_ont_color = "#45B29D"

hprc_color = "#FFE11A"
hprc_outer_color = "#FFF22B"

In [ ]:
# adjustText is used to keep point labels from overlapping.
pip install adjustText

In [ ]:
# PC1 vs PC2, with all participants in grey and long-read samples overlaid.
from adjustText import adjust_text

plt.rcParams['figure.figsize'] = [10, 10]
plt.rcParams.update({'font.size': 15})

# Create a scatter plot for each population in the full dataset
for pop, color in grey_map.items():
    # Filter points belonging to the current population
    indices = [i for i, item in enumerate(l1) if item == pop]
    plt.scatter(
        [pc1[i] for i in indices],
        [pc2[i] for i in indices],
        color=color,
        alpha=0.1,
        s=2,
        label=pop  # Label for legend
    )

a1 = plt.scatter([pb_pc1[0]], [pb_pc2[0]], marker="D", edgecolor=outer_color, color=pb_color, s=20, label='PacBio (N=1,027)')
a2 = plt.scatter([ont_pc1[0]], [ont_pc2[0]], marker="D", edgecolor=outer_color, color="#DF5A49", s=20, label='PacBio and ONT (N=50)')
a3 = plt.scatter([hprc_pc1[0]], [hprc_pc2[0]], marker="^", edgecolor="#36B1BF", color="#4AD9D9", s=30, label='HPRC (N=47)')
    
plt.scatter(
    pb_pc1,
    pb_pc2,
    marker="D",
    edgecolor=outer_color,
    color=pb_color,
    s=20
)

plt.scatter(
    ont_pc1,
    ont_pc2,
    marker="D",
    edgecolor=outer_color,
    color="#DF5A49",
    s=20
)
        
plt.scatter(
    hprc_pc1,
    hprc_pc2,
    marker="^",
    edgecolor="#36B1BF",
    color="#4AD9D9",
    s=30
)
    
plt.xlabel(f"PC1 ({100.0*variance_explained[0]:.2f}%)")
plt.ylabel(f"PC2 ({100.0*variance_explained[1]:.2f}%)")

legend2 = plt.legend(handles=[a1, a2, a3], loc='lower left')
plt.gca().add_artist(legend2)

plt.savefig('pca_2d_grey.proj.pdf', format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
# PC1 vs PC2, with all participants coloured by predicted ancestry.
plt.rcParams['figure.figsize'] = [10, 10]
plt.rcParams.update({'font.size': 15})

# Create a scatter plot for each population in the full dataset
for pop, color in color_map.items():
    # Filter points belonging to the current population
    indices = [i for i, item in enumerate(l1) if item == pop]
    plt.scatter(
        [pc1[i] for i in indices],
        [pc2[i] for i in indices],
        color=color,
        alpha=0.01,
        s=2,
        label=pop  # Label for legend
    )

# a1 = plt.scatter([pb_pc1[0]], [pb_pc2[0]], marker="D", edgecolor=outer_color, color=pb_color, s=20, label='PacBio (N=1,027)')
# a2 = plt.scatter([ont_pc1[0]], [ont_pc2[0]], marker="D", edgecolor=outer_color, color=pb_ont_color, s=20, label='PacBio and ONT (N=50)')
# a3 = plt.scatter([hprc_pc1[0]], [hprc_pc2[0]], marker="^", edgecolor=hprc_outer_color, color=hprc_color, s=20, label='HPRC (N=47)')

a1 = plt.scatter([pb_pc1[0]], [pb_pc2[0]], marker="D", edgecolor=outer_color, color=pb_color, s=20, label='PacBio (N=1,027)')
a2 = plt.scatter([ont_pc1[0]], [ont_pc2[0]], marker="D", edgecolor=outer_color, color="#DF5A49", s=20, label='PacBio and ONT (N=50)')
a3 = plt.scatter([hprc_pc1[0]], [hprc_pc2[0]], marker="^", edgecolor="#36B1BF", color="#4AD9D9", s=30, label='HPRC (N=47)')

plt.scatter(
    pb_pc1,
    pb_pc2,
    marker="D",
    edgecolor=outer_color,
    color=pb_color,
    s=20
)

plt.scatter(
    ont_pc1,
    ont_pc2,
    marker="D",
#     edgecolor=outer_color,
#     color=pb_ont_color,
    edgecolor=outer_color,
    color="#DF5A49",
    s=20
)
        
plt.scatter(
    hprc_pc1,
    hprc_pc2,
    marker="^",
#     edgecolor=hprc_outer_color,
#     color=hprc_color,
    edgecolor="#36B1BF",
    color="#4AD9D9",    
    s=30
)

plt.xlabel(f"PC1 ({100.0*variance_explained[0]:.2f}%)")
plt.ylabel(f"PC2 ({100.0*variance_explained[1]:.2f}%)")

legend1_patches = [mpatches.Patch(color=color, label=pop.upper()) for pop, color in color_map.items()]
legend1 = plt.legend(handles=legend1_patches)
plt.gca().add_artist(legend1)

legend2 = plt.legend(handles=[a1, a2, a3], loc='lower left')
plt.gca().add_artist(legend2)

plt.savefig('pca_2d.proj.pdf', format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
# The same plot with larger fonts, for the figure.
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = [10, 10]
plt.rcParams.update({'font.size': 22})  # Increased base font size

# Create a scatter plot for each population in the full dataset
for pop, color in color_map.items():
    indices = [i for i, item in enumerate(l1) if item == pop]
    plt.scatter(
        [pc1[i] for i in indices],
        [pc2[i] for i in indices],
        color=color,
        alpha=0.01,
        s=2,
        label=pop
    )

# Key points
a1 = plt.scatter([pb_pc1[0]], [pb_pc2[0]], marker="D", edgecolor=outer_color, color=pb_color, s=60, label='PacBio (N=1,027)')
a2 = plt.scatter([ont_pc1[0]], [ont_pc2[0]], marker="D", edgecolor=outer_color, color="#DF5A49", s=60, label='PacBio and ONT (N=50)')
a3 = plt.scatter([hprc_pc1[0]], [hprc_pc2[0]], marker="^", edgecolor="#36B1BF", color="#4AD9D9", s=80, label='HPRC (N=47)')

# Full datasets
plt.scatter(pb_pc1, pb_pc2, marker="D", edgecolor=outer_color, color=pb_color, s=30)
plt.scatter(ont_pc1, ont_pc2, marker="D", edgecolor=outer_color, color="#DF5A49", s=30)
plt.scatter(hprc_pc1, hprc_pc2, marker="^", edgecolor="#36B1BF", color="#4AD9D9", s=40)

# Axis labels
plt.xlabel(f"PC1 ({100.0*variance_explained[0]:.2f}%)", fontsize=30)
plt.ylabel(f"PC2 ({100.0*variance_explained[1]:.2f}%)", fontsize=30)

# Swapped legend positions
legend2 = plt.legend(handles=[a1, a2, a3], loc='upper left', fontsize=18, title="Long-read Samples", title_fontsize=16)
plt.gca().add_artist(legend2)

legend1_patches = [mpatches.Patch(color=color, label=pop.upper()) for pop, color in color_map.items()]
legend1 = plt.legend(handles=legend1_patches, loc='lower left', fontsize=18, title="Population", title_fontsize=16)
plt.gca().add_artist(legend1)

plt.tight_layout()
plt.savefig('pca_2d.proj.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side PC1 vs PC2 and PC2 vs PC3 panels.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
colors = [color_map[item] for item in l1]

# Create a scatter plot for each population in the full dataset
for pop, color in color_map.items():
    # Filter points belonging to the current population
    indices = [i for i, item in enumerate(l1) if item == pop]
    ax1.scatter(
        [pc1[i] for i in indices],
        [pc2[i] for i in indices],
        color=color,
        alpha=0.1,
        s=2,
        label=pop  # Label for legend
    )

a1 = ax1.scatter([pb_pc1[0]], [pb_pc2[0]], marker="D", edgecolor=outer_color, color=pb_color, s=20, label='PacBio (N=1,027)')
a2 = ax1.scatter([ont_pc1[0]], [ont_pc2[0]], marker="D", edgecolor=outer_color, color=pb_ont_color, s=20, label='PacBio and ONT (N=50)')
a3 = ax1.scatter([hprc_pc1[0]], [hprc_pc2[0]], marker="^", edgecolor=hprc_outer_color, color=hprc_color, s=30, label='HPRC (N=47)')
    
ax1.scatter(
    pb_pc1,
    pb_pc2,
    marker="D",
    color=pb_color,
    edgecolor=outer_color,
    alpha=0.9,
    s=15
)

ax1.scatter(
    ont_pc1,
    ont_pc2,
    marker="D",
    color=pb_ont_color,
    edgecolor=outer_color,
    alpha=0.9,
    s=20
)

ax1.scatter(
    hprc_pc1,
    hprc_pc2,
    marker="^",
    color=hprc_color,
    edgecolor=hprc_outer_color,
    alpha=0.9,
    s=30
)

ax1.set_xlabel(f"PC1 ({100.0*variance_explained[0]:.2f}%)", labelpad=10)
ax1.set_ylabel(f"PC2 ({100.0*variance_explained[1]:.2f}%)", labelpad=10)

# Create a scatter plot for each population in the full dataset
for pop, color in color_map.items():
    # Filter points belonging to the current population
    indices = [i for i, item in enumerate(l1) if item == pop]
    plt.scatter(
        [pc2[i] for i in indices],
        [pc3[i] for i in indices],
        color=color,
        alpha=0.1,
        s=2,
        label=pop  # Label for legend
    )

a1 = ax2.scatter([pb_pc2[0]], [pb_pc3[0]], marker="D", edgecolor=outer_color, color=pb_color, s=20, label='PacBio (N=1,027)')
a2 = ax2.scatter([ont_pc2[0]], [ont_pc3[0]], marker="D", edgecolor=outer_color, color=pb_ont_color, s=20, label='PacBio and ONT (N=50)')
a3 = ax2.scatter([hprc_pc2[0]], [hprc_pc3[0]], marker="^", edgecolor=hprc_outer_color, color=hprc_color, s=30, label='HPRC (N=47)')

ax2.scatter(
    pb_pc2,
    pb_pc3,
    marker="D",
    color=pb_color,
    edgecolor=outer_color,
    alpha=0.9,
    s=20
)

ax2.scatter(
    ont_pc2,
    ont_pc3,
    marker="D",
    color=pb_ont_color,
    edgecolor=outer_color,
    alpha=0.9,
    s=20
)

ax2.scatter(
    hprc_pc2,
    hprc_pc3,
    marker="^",
    color=hprc_color,
    edgecolor=hprc_outer_color,
    alpha=0.9,
    s=30
)

ax2.set_xlabel(f"PC2 ({100.0*variance_explained[1]:.2f}%)", labelpad=10)
ax2.set_ylabel(f"PC3 ({100.0*variance_explained[2]:.2f}%)", labelpad=10)

legend_patches = [mpatches.Patch(color=color, label=pop.upper()) for pop, color in color_map.items()]
legend1 = ax2.legend(handles=legend_patches)
plt.gca().add_artist(legend1)

legend2 = ax2.legend(handles=[a1, a2, a3], loc='upper center')
plt.gca().add_artist(legend2)

plt.tight_layout()

plt.savefig('pca_two_panel.proj.pdf', format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
# 3-D plot of PC1-PC3.
%matplotlib inline

fig = plt.figure()
ax = plt.axes(projection='3d')
ax.set_aspect("equal")

#ax.scatter3D(pc1, pc2, pc3, s=2, marker='o', depthshade=False, c=[f'{color_map[item]}08' for item in l])

# Create a scatter plot for each population in the full dataset
for pop, color in color_map.items():
    # Filter points belonging to the current population
    indices = [i for i, item in enumerate(l1) if item == pop]
    ax.scatter3D(
        [pc1[i] for i in indices],
        [pc2[i] for i in indices],
        [pc3[i] for i in indices],
        s=2,
        marker='o',
        depthshade=False,
        color=color,
        alpha=0.1,
        label=pop  # Label for legend
    )

ax.set_xlabel(f"PC1 ({100.0*variance_explained[0]:.2f}%)", labelpad=10)
ax.set_ylabel(f"PC2 ({100.0*variance_explained[1]:.2f}%)", labelpad=10)
ax.set_zlabel(f"PC3 ({100.0*variance_explained[2]:.2f}%)", labelpad=10)

ax.view_init(30, -140, 0)

a1 = ax.scatter3D([pb_pc1[0]], [pb_pc2[0]], [pb_pc3[0]], s=20, depthshade=False, marker="D", color=pb_color, edgecolor=outer_color, label='PacBio (N=1,027)')
a2 = ax.scatter3D([ont_pc1[0]], [ont_pc2[0]], [ont_pc3[0]], s=20, depthshade=False, marker="D", color=pb_ont_color, edgecolor=outer_color, label='PacBio and ONT (N=50)')
a3 = ax.scatter3D([hprc_pc1[0]], [hprc_pc2[0]], [hprc_pc3[0]], s=30, depthshade=False, marker="^", color=hprc_color, edgecolor=hprc_outer_color, label='HPRC (N=47)')

ax.scatter3D(pb_pc1, pb_pc2, pb_pc3, s=20, depthshade=False, marker='D', color=pb_color, edgecolor=outer_color, alpha=0.9)
ax.scatter3D(ont_pc1, ont_pc2, ont_pc3, s=20, depthshade=False, marker='s', color=pb_ont_color, edgecolor=outer_color, alpha=0.9)
ax.scatter3D(hprc_pc1, hprc_pc2, hprc_pc3, s=30, depthshade=False, marker='^', color=hprc_color, edgecolor=hprc_outer_color, alpha=0.9)

legend_patches = [mpatches.Patch(color=color, label=pop.upper()) for pop, color in color_map.items()]
legend1 = ax.legend(handles=legend_patches, loc=(0.15, 0.65))
plt.gca().add_artist(legend1)

legend2 = ax.legend(handles=[a1, a2, a3], loc=(0.55, 0.85))
plt.gca().add_artist(legend2)

plt.savefig('pca_3d.proj.pdf', format='pdf', bbox_inches='tight')

plt.show()